# Encycolpedia of Melbourne Tutorial

The [Encyclopedia of Melbourne](https://www.emelbourne.net.au/) is an online Encyclopedia made up of 1,613 entries detailing the history of Melbourne on different themes, as well as summaries of 1,241 key figures and links to their entries in the Australian Dictionary of Biography. This data is also saved as an RO-Crate. 

This tutorial will extend that of [Exploring temporal dimensions of RO-Crates
](https://automatic-carnival-o7wg71n.pages.github.io/tutorials/exploring-temporal-dimensions/) to build a comparative timeline of the people included in the Encyclopedia of Melbourne, and explore what analysis we can take from it. 

## What you'll learn


## Running this tutorial
While `crategraph` is pre-release, launch from the repository root with `uv run`, pulling in the project plus the plotting dependencies:
```
uv run --all-extras --with jupyter --with pandas --with plotly --with kaleido jupyter notebook
```

## 1. Load the crate and get an overview

In [ ]:
from crategraph import Crate

crate = Crate(
    "./data/EMEL"
)
crate

In [ ]:
crate.summary()

In [ ]:
crate.glimpse()

Reviewing the summary and the glimpse, we can see that they largely fall into
- Persons, with many smaller associated alternative name options
- Places
- Entries
- Repository objects and their connected files, connecting to all three. 

## 2. Formatting Dates
In [Exploring temporal dimensions of RO-Crates
](https://automatic-carnival-o7wg71n.pages.github.io/tutorials/exploring-temporal-dimensions/) we learnt how to use Crategraph to read, format, and save the dates that are included in different entities, and created a timeline across different entitity types. We will use that code to see how dates are used in this dataset:

In [ ]:
import pandas as pd
import plotly.express as px

types = ["Person", "Entry", "Event", "Place", "Theme", "RepositoryObject"]
dated = crate.select(entity_types=types).convert_dates(report=True)

comparison_df = pd.DataFrame(
    dated.entity_records(
        columns=["label", "type", "start_date", "end_date", "date_precision", "year"]
    )
)

fig = px.scatter(
    comparison_df,
    x="start_date",
    y="type",
    color="type",
    hover_name="label",
    hover_data={"year": True, "date_precision": True, "type": False, "start_date": False},
    category_orders={"type": types},
    title="University of Melbourne records through time, by type",
)
fig.update_traces(marker=dict(size=11, opacity=0.6))
fig.update_layout(yaxis_title=None, xaxis_title="Year", legend_title=None, height=400)
fig.show()

Here we can see we have meaningful dates for many Person and Repoistory Object entities, but only one for an Entry and none for Events and Places. We can also check the precison on our dates:

In [ ]:
comparison_df["date_precision"].value_counts()

And see that in almost every case, we have dates precise to the day. 

## 3. Tidying the Persons data
Let's delve into the people included in the Encycolpedia of Melbourne, starting by creating a dataframe with formatted dates:

In [ ]:
people = crate.select(entity_types=["Person"]).convert_dates(report=False)
df = pd.DataFrame(people.entity_records())

We can rename the start and end date columns to something that's more meaningful for people (birth and death). 

In [ ]:
df = df.rename(columns={"start_date": "birth_date", "end_date": "death_date"})

We can convert any empty lists to be n/a (for this we will need to import another package, Numpy), and drop any columns where everything is n/a: 

In [ ]:
import numpy as np

df = df.map(lambda x: np.nan if isinstance(x, list) and len(x) == 0 else x)
df = df.dropna(how="all", axis=1)

And lets tidy up some columns where the data is exactly the same:

In [ ]:
if df["label"].equals(df["name"]):
    df = df.drop("label", axis=1)

if df["type"].equals(
    df["types"].map(lambda x: x[0] if isinstance(x, list) and len(x) == 1 else x)
):
    df = df.drop("types", axis=1)

By default, Pandas will only show the first and last columns when there are many columns in a dataframe. You can change this to be able to see them all:

In [ ]:
pd.set_option("display.max_columns", None)

In [ ]:
df = df[
    [
        "id",
        "identifier",
        "name",
        "alsoKnownAs",
        "type",
        "birthPlace",
        "birthState",
        "deathPlace",
        "deathState",
        "birth_date",
        "startDate",
        "startDateModifier",
        "startDateISOString",
        "death_date",
        "endDate",
        "endDateModifier",
        "endDateISOString",
        "date_precision",
        "date_circa",
        "date_uncertain",
        "function",
        "gender",
        "summaryNote",
        "year",
        "recordAppendDate",
        "sourceOf",
        "online",
        "epub",
        "gallery",
        "targetOf",
    ]
]

## 3. Exploring the Persons data
There seem to be some people that only have an id, a name, and a type. Let's check how many:

In [ ]:
cols_to_check = df.columns.difference(["id", "name", "type"])
rows_all_na = (df[cols_to_check].isna().all(axis=1)).sum()

print(f"Rows with NA in all columns except 'id' and 'name': {rows_all_na}")

In [ ]:
df.loc[df["identifier"].isna()]

Referencing the Encyclopedia, it appears that these are authors of Encyclopedia entries, not subjects included. Lets split them out into their own dataframe:

In [ ]:
authors = df.loc[df["identifier"].isna()].copy().reset_index(drop=True)
df = df.loc[df["identifier"].notna()].copy().reset_index(drop=True)

We can then check and confirm we have a birth and death date for all our remaining people:

In [ ]:
print(
    f"There are {df.loc[df['death_date'].isna()].shape[0]} people with no birth_date recorded and {df.loc[df['death_date'].isna()].shape[0]} people with no death date recorded."
)

## 4. Visualisaing Lifetimes

Plotly has a timeline function, that uses a gantt chart to create a timeline where we can record people's lives from birth to death. Let's start with just the basic information that we need:
- our data source: df
- x_start: when the person's lifeline begins
- x_end: when the person's lifeline ends
- title: Describing what we are looking at

In [ ]:
fig = px.timeline(
    df,
    x_start="birth_date",
    x_end="death_date",
    title="Lifetimes of People Included in the Encyclopedia of Melbourne",
    hover_data={
        "name": True,
        "birth_date": True,
        "death_date": True,
    },
)

fig.show()

This is a little hard to read! Lets start by increasing the height of our plot, by specifying a height value. Note, you can also use the controls at the top of the Plotly figure to zoom in and around:

In [ ]:
fig = px.timeline(
    df,
    x_start="birth_date",
    x_end="death_date",
    hover_data={
        "name": True,
        "birth_date": True,
        "death_date": True,
    },
    title="Lifetimes of People Included in the Encyclopedia of Melbourne",
    height=1800,
)

fig.show()

We can also change the order the plots appear on the graph, so it doesn't look quite so scattershot and get a better idea of the timeframes of the people we are looking at, by sorting by date of birth and death:

In [ ]:
df = df.sort_values(by=["birth_date", "death_date"]).reset_index(drop=True)

In [ ]:
fig = px.timeline(
    df,
    x_start="birth_date",
    x_end="death_date",
    hover_data={
        "name": True,
        "birth_date": True,
        "death_date": True,
    },
    title="Lifetimes of People Included in the Encyclopedia of Melbourne",
    height=1800,
)

fig.show()

This chart shows a focus on the eighteenth and nineteenth century - it's only as we get to the top 10% that we see birthdates in the twentieth century. We can confirm this with our data source: 

In [ ]:
# Define century boundaries
bins = [
    pd.Timestamp("1700-01-01"),
    pd.Timestamp("1800-01-01"),
    pd.Timestamp("1900-01-01"),
    pd.Timestamp("2000-01-01"),
]
labels = ["18th century", "19th century", "20th century"]

df["century"] = pd.cut(df["birth_date"], bins=bins, labels=labels, right=False)

for l in labels:
    print(f"There are {df.loc[df['century'] == l].shape[0]} people born in the {l}")

And by plotting a histogram:

In [ ]:
fig = px.histogram(df, x="birth_date")
fig.show()

We can also make the chart more interesting by using other data in our dataset, such colour coding by gender:

In [ ]:
fig = px.timeline(
    df,
    x_start="birth_date",
    x_end="death_date",
    hover_data={
        "name": True,
        "birth_date": True,
        "death_date": True,
    },
    title="Lifetimes of People Included in the Encyclopedia of Melbourne, coded by gender",
    height=1800,
    color="gender",
)

fig.show()

This shows a stark difference in the number of women vs men included in the Encyclopedia! Lets get exact numbers:

In [ ]:
df["gender"].value_counts()

Because we have not specifed a y-axis location, Plotly has started both from the bottom up, but this means some of the data is covered. We can use our index, which is a sequence of unique numbers, to specify where we want each lifeline to go - we just need to create it as a new column:

In [ ]:
df = df.reset_index().rename(columns={"index": "y-axis_order"})

In [ ]:
fig = px.timeline(
    df,
    x_start="birth_date",
    x_end="death_date",
    y="y-axis_order",
    hover_data={
        "name": True,
        "birth_date": True,
        "death_date": True,
    },
    title="Lifetimes of People Included in the Encyclopedia of Melbourne",
    height=1800,
    color="gender",
)

fig.show()